#### Before running this file, download the TCGA data in the `download_UCSCtcga_data.ipynb`

In [1]:
import pandas as pd
import gzip
import polars as pl
import os
import requests
import shutil

# Create RNA seq normal samples DF

In [6]:
def download_data(download_endpoint, output_file_path):
    decompressed_file_path = output_file_path.replace('.gz', '')
    if os.path.exists(decompressed_file_path):
        print(f"File already exists: {decompressed_file_path}")
        return

    response = requests.get(download_endpoint, stream=True)

    if response.status_code == 200:
        print(f"Starting the download")
        os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
        with open(output_file_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)

        print(f"Downloaded file saved to {output_file_path}")

        with gzip.open(output_file_path, 'rb') as f_in:
            with open(output_file_path.replace('.gz', ''), 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)

        print(f"Decompressed file saved to {output_file_path.replace('.gz', '')}")

        os.remove(output_file_path)
        print(f"Removed compressed file: {output_file_path}")
    else:
        print(f"Failed to download file. Status code: {response.status_code}")

In [ ]:
download_endpoint = "https://toil-xena-hub.s3.us-east-1.amazonaws.com/download/TcgaTargetGtex_gene_expected_count.gz"
output_file_path = './toil_expected_count.gz'
download_data(download_endpoint, output_file_path)

Starting the download
Downloaded file saved to ./toil_expected_count.gz
Decompressed file saved to ./toil_expected_count
Removed compressed file: ./toil_expected_count.gz


In [9]:
download_endpoint = "https://toil-xena-hub.s3.us-east-1.amazonaws.com/download/TcgaTargetGTEX_phenotype.txt.gz"
output_file_path = './toil_phenotype.gz'
download_data(download_endpoint, output_file_path)

Starting the download
Downloaded file saved to ./toil_phenotype.gz
Decompressed file saved to ./toil_phenotype
Removed compressed file: ./toil_phenotype.gz


# Create DNA methylation normal samples DF

In [ ]:
# Downloaded from 'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE66nnn/GSE66351/matrix/'
filepath = './GSE66351_series_matrix.txt.gz'

sample_titles = []
skip_lines = 0

with gzip.open(filepath, 'rt') as f:
    for line in f:
        if line.startswith('!Sample_title'):
            raw_titles = line.strip().split('\t')[1:] # Skip the word '!Sample_title'
            sample_titles = [title.strip('"') for title in raw_titles]
        
        # Stop counting when table starts 
        if line.startswith('"ID_REF"'):
            break
            
        skip_lines += 1


df = pd.read_csv(filepath, sep='\t', skiprows=skip_lines, index_col=0)
df.columns = sample_titles

# Retain only columns that contain both "CTRL" and "Glia"
normal_glia_cols = [col for col in df.columns if 'CTRL' in col and 'Glia' in col]
normal_samples_df = df[normal_glia_cols]

print(normal_samples_df.shape)
print(normal_samples_df.head())

(481869, 16)
            Dnr06_M_18_CTRL_Glia  Dnr13_F_77_CTRL_Glia  Dnr14_F_77_CTRL_Glia  \
ID_REF                                                                         
cg00000029               0.59214               0.44301               0.43959   
cg00000108               0.92945               0.90099               0.90670   
cg00000109               0.85031               0.81780               0.84732   
cg00000165               0.17756               0.18894               0.13824   
cg00000236               0.81965               0.74335               0.68259   

            Dnr15_F_83_CTRL_Glia  Dnr16_F_88_CTRL_Glia  Dnr17_M_19_CTRL_Glia  \
ID_REF                                                                         
cg00000029               0.45112               0.46215               0.39536   
cg00000108               0.91448               0.92725               0.91978   
cg00000109               0.83608               0.89326               0.80183   
cg00000165               0

In [9]:
print("Total missing values:", normal_samples_df.isna().sum().sum())
normal_samples_df_clean = normal_samples_df.dropna()
print(f"Probes remaining after dropping NAs: {normal_samples_df_clean.shape[0]}")

Total missing values: 16
Probes remaining after dropping NAs: 481868


In [ ]:
tcga_dna_methyl_file_path = 'UCSC_TCGA_Data_Clean/Clean/dna_clean.tsv'
tcga_probes = pl.read_csv(tcga_dna_methyl_file_path, separator='\t', columns=['Probe_ID'])['Probe_ID'].to_list()

tcga_probes_set = set(tcga_probes)
geo_intersected_df = normal_samples_df_clean[normal_samples_df_clean.index.isin(tcga_probes_set)]
geo_normal_samples_final_df = geo_intersected_df.reset_index().rename(columns={'ID_REF': 'Probe_ID'})
print(f"GEO shape after intersecting with TCGA: {geo_normal_samples_final_df.shape}")

output_geo_path = 'UCSC_TCGA_Data_Clean/Clean/geo_normal_glia_clean.tsv'
geo_normal_samples_final_df.to_csv(output_geo_path, sep='\t', index=False)
print(f"GEO normal data successfully saved to {output_geo_path}")

GEO shape after intersecting with TCGA: (29677, 17)
Success! Intersected GEO normal data saved to UCSC_TCGA_Data_Clean/Clean/geo_normal_glia_clean.tsv


In [4]:
y_minus_1 = [
    46,    # TCGA-ACC
    343,   # TCGA-BLCA
    919,   # TCGA-BRCA
    172,   # TCGA-CESC
    30,    # TCGA-CHOL
    363,   # TCGA-COAD
    33,    # TCGA-DLBC
    126,   # TCGA-ESCA
    243,   # TCGA-GBM
    354,   # TCGA-HNSC
    63,    # TCGA-KICH
    478,   # TCGA-KIRC
    216,   # TCGA-KIRP
    # TCGA-LAML skipped (no data)
    435,   # TCGA-LGG
    184,   # TCGA-LIHC
    365,   # TCGA-LUAD
    328,   # TCGA-LUSC
    62,    # TCGA-MESO
    432,   # TCGA-OV
    120,   # TCGA-PAAD
    82,    # TCGA-PCPG
    352,   # TCGA-PRAD
    132,   # TCGA-READ
    226,   # TCGA-SARC
    352,   # TCGA-SKCM
    357,   # TCGA-STAD
    122,   # TCGA-TGCT
    379,   # TCGA-THCA
    90,    # TCGA-THYM
    440,   # TCGA-UCEC
    48,    # TCGA-UCS
    12,    # TCGA-UVM
]

print(f"Number of projects: {len(y_minus_1)}")  # 32
print(f"Total samples: {sum(y_minus_1)}")        # 7904

Number of projects: 32
Total samples: 7904
